In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint

# [LOG] Model Versioning

## Version 1: Tiny-Baseline 
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

----

# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [2]:
# ==============================================================================
# IL GENERATORE DI DATI (DATA ENGINE)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=2, alpha=0.05, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_coords_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   
            people_xy = data['people_xy']   
            people_mask = data['people_mask'] 
            T = raw_iq.shape[0]             
            
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            for t in range(T):
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
            X_batch.append(decluttered)

            y_coords_batch.append(people_xy.reshape(T, 8))
            y_mask_batch.append(people_mask)

       # X = np.concatenate(X_batch, axis=0)
       # Y_coords = np.concatenate(y_coords_batch, axis=0)
       # Y_mask = np.concatenate(y_mask_batch, axis=0)
       # return X, {"coords_head": Y_coords, "mask_head": Y_mask}
        X = np.concatenate(X_batch, axis=0).astype(np.float32)       # <--- AGGIUNTO astype
        Y_coords = np.concatenate(y_coords_batch, axis=0).astype(np.float32) # <--- AGGIUNTO astype
        Y_mask = np.concatenate(y_mask_batch, axis=0).astype(np.float32)   # <--- AGGIUNTO astype
        return X, {"coords_head": Y_coords, "mask_head": Y_mask}
       
       

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_paths)

# --- INIZIALIZZAZIONE ---
train_indices = [22, 0, 1, 2, 3, 10, 14, 16, 17, 18, 19, 21, 8, 9, 12, 5, 6, 4]
val_indices = [23, 7, 11, 13, 15, 20]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

BATCH_SIZE = 4 
train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=False)

print(f"Motore pronto: {len(train_files)} file di Train, {len(val_files)} file di Validation.")

Motore pronto: 18 file di Train, 6 file di Validation.


In [3]:
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE). 
    In futuro potremo azzerarlo se la maschera è 0.
    """
    return losses.mean_squared_error(y_true, y_pred)

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 
# ==============================================================================
def build_eeai_model_v2(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # Layer più larghi! 32 invece di 16, 64 invece di 32.
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_base")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) 

    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 

    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # Collo di bottiglia allargato
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_v2")

model_v2 = build_eeai_model_v2()
print("\n--- ARCHITETTURA V2 ---")
model_v2.summary()

# Compilazione
model_v2.compile(
    optimizer='adam',
    loss={"coords_head": "mse", "mask_head": "binary_crossentropy"},
    loss_weights={"coords_head": 1.0, "mask_head": 0.5}
)

# Salviamo un file col nome V2 per non sovrascrivere la V1!
checkpoint_v2 = ModelCheckpoint("eeai_best_model_v2.keras", monitor="val_loss", save_best_only=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 
print("\n--- INIZIO ADDESTRAMENTO---")
history_v2 = model_v2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint_v2],
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [4]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 E ADDESTRAMENTO OTTIMIZZATO
# ==============================================================================
def build_eeai_model_v2(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # Qui ci sono i tuoi layer convoluzionali
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_base")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) 
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    common_feat = layers.Dense(64, activation='relu', name="features")(x)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_v2")

model_v2 = build_eeai_model_v2()
model_v2.compile(
    optimizer='adam',
    loss={"coords_head": "mse", "mask_head": "binary_crossentropy"},
    loss_weights={"coords_head": 1.0, "mask_head": 0.5}
)
checkpoint_v2 = ModelCheckpoint("eeai_best_model_v2.keras", monitor="val_loss", save_best_only=True, verbose=1)

# --- IL "PONTE" PER MAC (Da inserire qui dentro, prima del .fit) ---
def make_mac_dataset(generator):
    return tf.data.Dataset.from_generator(
        lambda: generator,
        output_signature=(
            tf.TensorSpec(shape=(None, 1, 120, 18), dtype=tf.float32),
            {
                "coords_head": tf.TensorSpec(shape=(None, 8), dtype=tf.float32),
                "mask_head": tf.TensorSpec(shape=(None, 4), dtype=tf.float32)
            }
        )
    ).cache().prefetch(tf.data.AUTOTUNE)

train_dataset = make_mac_dataset(train_gen)
val_dataset = make_mac_dataset(val_gen)

# FUOCO ALLE POLVERI
EPOCHS = 50 
print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO ---")
history_v2 = model_v2.fit(
    train_dataset,                # Usiamo il dataset ponte
    validation_data=val_dataset,  # Usiamo il dataset ponte
    epochs=EPOCHS,
    callbacks=[checkpoint_v2],
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

2026-05-20 00:31:10.917247: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-05-20 00:31:10.917512: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-20 00:31:10.917538: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
I0000 00:00:1779229870.918521   47850 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1779229870.918605   47850 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



--- INIZIO ADDESTRAMENTO OTTIMIZZATO ---
Epoch 1/50


2026-05-20 00:31:11.874402: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


      5/Unknown 10s 2s/step - coords_head_loss: 11.0354 - loss: 11.8049 - mask_head_loss: 1.3357

2026-05-20 00:31:21.411827: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-05-20 00:31:21.412023: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/adam/Add_16/ReadVariableOp/_9]]
2026-05-20 00:31:21.412036: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 17520739503076884719
2026-05-20 00:31:21.412038: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 12865519370092118034
2026-05-20 00:31:21.412044: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 3623470123588280689
2026-05-20 00:31:21.412047: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key 

...

2026-05-20 00:31:34.136226: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 17520739503076884719
2026-05-20 00:31:34.136475: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 3623470123588280689
2026-05-20 00:31:34.136480: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 5656770681239085689
2026-05-20 00:31:34.136485: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 17130784654236452217
2026-05-20 00:31:34.136499: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 16312303938594433227
2026-05-20 00:31:34.136503: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 1603425302569913249
2026-05-20 00:31:34.136506: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv 


Epoch 3: val_loss improved from 4.90689 to 4.85727, saving model to eeai_best_model_v2.keras

Epoch 3: finished saving model to eeai_best_model_v2.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - coords_head_loss: 3.2826 - loss: 4.6268 - mask_head_loss: 1.0887 - val_coords_head_loss: 2.5124 - val_loss: 4.8573 - val_mask_head_loss: 0.5478
Epoch 4/50


2026-05-20 00:31:34.761896: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 6197429620865251951
2026-05-20 00:31:34.761926: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 18018240218334770196
2026-05-20 00:31:34.761929: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 10252953652787560028
2026-05-20 00:31:34.761932: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 11299158962343514002


1/5 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - coords_head_loss: 3.3240 - loss: 3.6887 - mask_head_loss: 0.7294

KeyboardInterrupt: 

### Come leggere le Loss del nostro Multi-Head Model

1. **coords_head_loss (L'Errore di Posizione - MSE)**
Misura la distanza matematica al quadrato. Esempio: se vale `3.35`, l'errore medio in metri è la radice quadrata (circa `1.83` metri). Più scende, più le X rosse si avvicinano ai pallini verdi.

2. **mask_head_loss (L'Errore di Presenza - Binary Crossentropy)**
Misura la confusione della rete sulla presenza o meno della persona (0 o 1). Più è bassa, meno fantasmi vedremo.

3. **loss (La Loss Totale)**
Il voto complessivo: `coords_loss * 1.0 + mask_loss * 0.5`. L'ottimizzatore cerca di abbassare questo numero il più possibile.

4. **val_loss (validation loss)**
È l'errore che la rete commette sui dati che non ha mai visto (l'esame di fine modulo).


In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.0 (Anti-Sfarfallio e Ground Truth Fixata)
# ==============================================================================
#file_target = "dataset/data/window_000007.npz"
file_target = "dataset/window_000007.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V2)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.05
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    # Usa esplicitamente il modello V2 appena addestrato!
    preds = model_v2.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V2 | Frame: {frame_idx}/{T-1} | Window: 07", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)